# Monthly Rainfall Analysis

This notebook cleans the hand-entered Machakos readings, uses NumPy to summarize monthly and annual rainfall across five counties, and recommends the strongest consecutive two-month planting windows.

## Data and method

Rainfall is measured in millimetres. Valid Machakos values are converted to integers; invalid entries are replaced by the rounded mean of valid readings. NumPy vector operations are used for county totals, monthly means, dry-month counts, and adjacent-month totals.

In [1]:
import csv
import numpy as np

counties = ['Nairobi', 'Nakuru', 'Kisumu', 'Mombasa', 'Machakos']
months = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],  # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],   # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],  # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],  # Mombasa
]
machakos_raw = ['48', '35', '88', 'N/A', '140', ' 20 ', '9', '12', '', '45', '182', '96']

def clean_machakos(raw_values, month_names):
    parsed = []
    imputed_months = []
    for month, raw_value in zip(month_names, raw_values):
        try:
            parsed.append(int(raw_value.strip()))
        except ValueError:
            parsed.append(np.nan)
            imputed_months.append(month)

    valid_mean = float(np.nanmean(parsed))
    rounded_mean = round(valid_mean)
    cleaned = np.where(np.isnan(parsed), rounded_mean, parsed).astype(np.int64)
    print(f'Machakos valid-value mean: {valid_mean:.1f} mm; rounded imputation: {rounded_mean} mm')
    print('Imputed months:', ', '.join(imputed_months))
    print('Cleaned Machakos rainfall (mm):', cleaned.tolist())
    return cleaned

machakos_clean = clean_machakos(machakos_raw, months)

Machakos valid-value mean: 67.5 mm; rounded imputation: 68 mm
Imputed months: April, September
Cleaned Machakos rainfall (mm): [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


In [2]:
R = np.array(clean_rows + [machakos_clean.tolist()], dtype=np.int64)
print('Combined rainfall array (mm):')
print(R)
print('Shape:', R.shape)
print('Dtype:', R.dtype)

Combined rainfall array (mm):
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Shape: (5, 12)
Dtype: int64


In [3]:
annual_totals = R.sum(axis=1)
monthly_averages = R.mean(axis=0)
print('Annual total rainfall (mm):')
print('\n'.join(f'{county}: {total}' for county, total in zip(counties, annual_totals)))
print('\nAverage rainfall by month across all counties (mm):')
print('\n'.join(f'{month}: {average:.1f}' for month, average in zip(months, monthly_averages)))

Annual total rainfall (mm):
Nairobi: 1045
Nakuru: 916
Kisumu: 1371
Mombasa: 1075
Machakos: 811

Average rainfall by month across all counties (mm):
January: 47.0
February: 47.6
March: 91.0
April: 164.4
May: 171.2
June: 67.6
July: 56.6
August: 58.2
September: 64.2
October: 67.8
November: 127.4
December: 80.6


In [4]:
wettest_indices = np.argmax(R, axis=1)
wettest_months = [months[index] for index in wettest_indices]
wettest_amounts = R[np.arange(len(counties)), wettest_indices]
print('Wettest month by county:')
for county, month, amount in zip(counties, wettest_months, wettest_amounts):
    print(f'{county}: {month} ({amount} mm)')

Wettest month by county:
Nairobi: April (219 mm)
Nakuru: April (145 mm)
Kisumu: April (240 mm)
Mombasa: May (240 mm)
Machakos: November (182 mm)


In [5]:
dry_month_mask = R < 50
dry_month_counts = np.count_nonzero(dry_month_mask, axis=1)
print('Dry months (rainfall below 50 mm):')
print('\n'.join(f'{county}: {count}' for county, count in zip(counties, dry_month_counts)))

Dry months (rainfall below 50 mm):
Nairobi: 3
Nakuru: 3
Kisumu: 0
Mombasa: 2
Machakos: 6


In [6]:
adjacent_totals = R[:, :-1] + R[:, 1:]
window_starts = np.argmax(adjacent_totals, axis=1)
planting_windows = [f'{months[index]}-{months[index + 1]}' for index in window_starts]
window_rainfall = adjacent_totals[np.arange(len(counties)), window_starts]
print('Recommended planting windows (highest adjacent-month rainfall):')
for county, window, total in zip(counties, planting_windows, window_rainfall):
    print(f'{county}: {window} ({total} mm)')

Recommended planting windows (highest adjacent-month rainfall):
Nairobi: April-May (395 mm)
Nakuru: April-May (265 mm)
Kisumu: April-May (420 mm)
Mombasa: April-May (390 mm)
Machakos: November-December (278 mm)


In [7]:
summary_rows = zip(counties, annual_totals, wettest_months, dry_month_counts, planting_windows)
with open('rainfall_summary.csv', 'w', newline='', encoding='utf-8') as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window'])
    writer.writerows(summary_rows)

print('Wrote rainfall_summary.csv with 5 county rows.')
with open('rainfall_summary.csv', newline='', encoding='utf-8') as csv_file:
    print(csv_file.read(), end='')

Wrote rainfall_summary.csv with 5 county rows.
county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,3,April-May
Nakuru,916,April,3,April-May
Kisumu,1371,April,0,April-May
Mombasa,1075,May,2,April-May
Machakos,811,November,6,November-December


## Planting Advisory

April–May is the highest combined-rainfall two-month period for Nairobi, Nakuru, Kisumu, and Mombasa, so treat it as an initial planting window rather than a guarantee. Machakos has its highest adjacent-month total in November–December, which is a provisional local signal for planting. Farmers should prepare land ahead of these periods and use current county forecasts and soil moisture to choose exact planting dates. Dry-month counts are highest in Machakos (six) and Nairobi and Nakuru (three each), highlighting the need to conserve water and consider drought-tolerant crops. The analysis summarizes one year of monthly station readings and cannot capture within-county variation or predict future rainfall.

## Data Quality Notes

The hand-entered Machakos row contained `N/A` for April and a blank for September. Both invalid values were replaced by the rounded mean of the ten valid readings (67.5 mm rounded to 68 mm), and the imputed months are printed during cleaning. These substitutions affect Machakos totals and its adjacent-month rainfall comparisons, so the resulting advisory should be treated cautiously and updated when verified station data become available.